**501328 ชาลินี  เจิ้ง**

In [ ]:
!pip install kaggle
!kaggle competitions download -c birdsong-recognition-cmu
!unzip birdsong-recognition-cmu.zip

In [ ]:
import torch
import os
import librosa
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from transformers import ASTFeatureExtractor, ASTForAudioClassification

# 1. ตั้งค่าพื้นฐาน
audio_train_path = "/content/Data_files/train"
audio_test_path = "/content/Data_files/test"
train_metadata_path = "/content/train.csv"
sample_submission_path = "/content/Sample_submission.csv"
output_submission_path = "Submission_5.csv"

# 2. โหลดข้อมูล CSV และเตรียม Label Encoding
dataframe = pd.read_csv(train_metadata_path)
label_encoder = LabelEncoder()
dataframe["encoded_genus"] = label_encoder.fit_transform(dataframe["genus"])
num_classes = len(label_encoder.classes_)

# 3. โหลด Feature Extractor และ Model
feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
classification_model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels=num_classes,
    ignore_mismatched_sizes=True  # Fix size mismatch issues
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classification_model.to(device)
classification_model.eval()

# 4. ฟังก์ชันแปลงเสียงเป็น Feature
def extract_audio_features(file_path):
    audio_data, sample_rate = librosa.load(file_path, sr=None)  # Load audio
    audio_data = librosa.resample(audio_data, orig_sr=sample_rate, target_sr=16000)  # Resample to 16kHz
    inputs = feature_extractor(audio_data, sampling_rate=16000, return_tensors="pt")
    return inputs.input_values[0]

# 📌 5. สร้าง Custom Dataset สำหรับ Training
class BirdSoundDataset(Dataset):
    def __init__(self, dataframe, audio_path, label_encoder, feature_extractor):
        self.dataframe = dataframe
        self.audio_path = audio_path
        self.label_encoder = label_encoder
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        file_id = row["file_id"]
        encoded_genus = row["encoded_genus"]
        file_path = os.path.join(self.audio_path, f"xc{file_id}.flac")

        if os.path.exists(file_path):
            features = extract_audio_features(file_path)
            return torch.tensor(features.numpy()), torch.tensor(encoded_genus)
        else:
            return None, None

train_dataset = BirdSoundDataset(dataframe, audio_train_path, label_encoder, feature_extractor)
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)

# 6. ทำ Fine-Tuning (Training)
optimizer = torch.optim.AdamW(classification_model.parameters(), lr=5e-5)
criterion = torch.nn.CrossEntropyLoss()

classification_model.train()
for epoch in range(10):
    epoch_loss = 0
    for batch in train_dataloader:
        inputs, labels = batch
        if inputs is None or labels is None:
            continue  # Skip invalid data

        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = classification_model(inputs).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1} Loss: {epoch_loss / len(train_dataloader):.4f}")




Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([14]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([14, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 Loss: 2.7409
Epoch 2 Loss: 1.0142
Epoch 3 Loss: 0.2684
Epoch 4 Loss: 0.0558
Epoch 5 Loss: 0.0212
Epoch 6 Loss: 0.0120
Epoch 7 Loss: 0.0079
Epoch 8 Loss: 0.0059
Epoch 9 Loss: 0.0048
Epoch 10 Loss: 0.0040


FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_submission.csv'

In [ ]:
# 7. ทำนายข้อมูล Test & สร้างไฟล์ Submission
submission_df = pd.read_csv('/content/Sample_submission.csv')

def predict_genus(file_path):
    feature = extract_audio_features(file_path).to(device)
    with torch.no_grad():
        output = classification_model(feature.unsqueeze(0)).logits
    predicted_index = torch.argmax(output, dim=1).item()
    return label_encoder.inverse_transform([predicted_index])[0]

for index, row in submission_df.iterrows():
    file_id = row["file_id"]
    file_path = os.path.join(audio_test_path, f"xc{file_id}.flac")

    if os.path.exists(file_path):
        predicted_label = predict_genus(file_path)
        submission_df.at[index, "genus"] = predicted_label

# 8. บันทึกไฟล์ Submission
submission_df.to_csv(output_submission_path, index=False)
print(f"📄 Submission file saved: {output_submission_path}")

📄 Submission file saved: Submission_5.csv
